# CSE445: Real-Time Phishing URL Detection Using Machine Learning

This notebook implements the project proposal using the **LegitPhish Dataset**.

**Class mapping**
- `0 = Phishing`
- `1 = Legitimate`

## Six models
1. Logistic Regression
2. SVM
3. Random Forest
4. XGBoost
5. K-Means Clustering
6. LLM-based tabular classifier


## 1. Install required packages

Run this cell in Google Colab.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib joblib xgboost transformers accelerate


## 2. Upload the LegitPhish CSV

Upload `url_features_extracted1.csv`.


In [ ]:
from google.colab import files
uploaded = files.upload()


## 3. Imports and configuration


In [ ]:
import io, os, re, math, json, warnings
from collections import Counter
from urllib.parse import urlparse, parse_qs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 4. Load and inspect dataset


In [ ]:
filename = next(iter(uploaded.keys()))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df.columns = [str(c).strip() for c in df.columns]

print("Shape:", df.shape)
display(df.head())
print(df.info())
print("\nClass distribution:")
print(df["ClassLabel"].value_counts().sort_index())


## 5. Cleaning and feature selection


In [ ]:
FEATURES = [
    "url_length", "has_ip_address", "dot_count", "https_flag",
    "url_entropy", "token_count", "subdomain_count", "query_param_count",
    "tld_length", "path_length", "has_hyphen_in_domain", "number_of_digits",
    "tld_popularity", "suspicious_file_extension",
    "domain_name_length", "percentage_numeric_chars"
]

TARGET = "ClassLabel"

df = df.drop_duplicates().copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype(int)

available_features = [c for c in FEATURES if c in df.columns]
for c in available_features:
    df[c] = pd.to_numeric(df[c], errors="coerce")

X = df[available_features]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Features used:", len(available_features))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 6. Evaluation helper


In [ ]:
results = []

def phishing_probability(model, X):
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)
        classes = list(model.classes_)
        return probs[:, classes.index(0)]
    if hasattr(model, "decision_function"):
        return -np.asarray(model.decision_function(X))
    return None

def evaluate(name, y_true, y_pred, score=None):
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_Phishing": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "Recall_Phishing": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "F1_Phishing": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "ROC_AUC": np.nan
    }

    if score is not None:
        row["ROC_AUC"] = roc_auc_score((np.asarray(y_true) == 0).astype(int), score)

    results.append(row)

    print("\\n", "="*65)
    print(name)
    print("="*65)
    print(classification_report(
        y_true, y_pred,
        labels=[0,1],
        target_names=["Phishing (0)", "Legitimate (1)"],
        digits=4
    ))

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Phishing","Legitimate"])
    disp.plot(values_format="d")
    plt.title(name + " - Confusion Matrix")
    plt.show()

    if score is not None:
        RocCurveDisplay.from_predictions(
            (np.asarray(y_true) == 0).astype(int), score
        )
        plt.title(name + " - ROC Curve")
        plt.show()

    return row


## 7. Model 1 — Logistic Regression


In [ ]:
lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])
lr.fit(X_train, y_train)
pred = lr.predict(X_test)
evaluate("Logistic Regression", y_test, pred, phishing_probability(lr, X_test))


## 8. Model 2 — Support Vector Machine (SVM)


In [ ]:
svm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE))
])
svm.fit(X_train, y_train)
pred = svm.predict(X_test)
evaluate("SVM", y_test, pred, phishing_probability(svm, X_test))


## 9. Model 3 — Random Forest


In [ ]:
rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
evaluate("Random Forest", y_test, pred, phishing_probability(rf, X_test))


## 10. Model 4 — XGBoost


In [ ]:
xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        objective="binary:logistic", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])
xgb.fit(X_train, y_train)
pred = xgb.predict(X_test)
evaluate("XGBoost", y_test, pred, phishing_probability(xgb, X_test))


## 11. Model 5 — K-Means Clustering

K-Means is unsupervised. We use two clusters and map each cluster to the majority real class from the training set.


In [ ]:
kmeans_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KMeans(n_clusters=2, n_init=20, random_state=RANDOM_STATE))
])

train_clusters = kmeans_pipe.fit_predict(X_train)
cluster_map = {}

y_train_np = y_train.to_numpy()
for c in [0, 1]:
    labels = y_train_np[train_clusters == c]
    counts = np.bincount(labels.astype(int), minlength=2)
    cluster_map[c] = int(np.argmax(counts))

test_clusters = kmeans_pipe.predict(X_test)
kmeans_pred = np.array([cluster_map[int(c)] for c in test_clusters])

Xt = kmeans_pipe[:-1].transform(X_test)
dist = kmeans_pipe.named_steps["model"].transform(Xt)

phish_clusters = [c for c, label in cluster_map.items() if label == 0]
legit_clusters = [c for c, label in cluster_map.items() if label == 1]

if phish_clusters and legit_clusters:
    dp = dist[:, phish_clusters[0]]
    dl = dist[:, legit_clusters[0]]
    kmeans_score = dl / (dp + dl + 1e-12)
else:
    kmeans_score = (kmeans_pred == 0).astype(float)

print("Cluster-to-class mapping:", cluster_map)
evaluate("K-Means Clustering", y_test, kmeans_pred, kmeans_score)


## 12. Compare the first five models


In [ ]:
comparison_5 = pd.DataFrame(results).sort_values(
    ["Recall_Phishing", "F1_Phishing", "ROC_AUC"],
    ascending=False
).reset_index(drop=True)

display(comparison_5.style.format({
    "Accuracy": "{:.4f}",
    "Precision_Phishing": "{:.4f}",
    "Recall_Phishing": "{:.4f}",
    "F1_Phishing": "{:.4f}",
    "ROC_AUC": "{:.4f}",
}))


## 13. Model 6 — Large Language Model (LLM)

For tabular data, every row is converted into a text description such as:

`url_length=120, has_ip_address=1, url_entropy=4.8, ...`

The instruction LLM then returns either `phishing` or `legitimate`.

The default is `Qwen/Qwen2.5-0.5B-Instruct`.  
To keep Colab runtime practical, evaluate a subset such as 100–200 test rows.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LLM_SAMPLES = 100   # Increase to 200 or more if you have enough runtime.

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

model_kwargs = {}
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16
    model_kwargs["device_map"] = "auto"

llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, **model_kwargs)
if not torch.cuda.is_available():
    llm = llm.to("cpu")

test_idx = X_test.sample(
    n=min(LLM_SAMPLES, len(X_test)),
    random_state=RANDOM_STATE
).index

X_llm = X_test.loc[test_idx]
y_llm = y_test.loc[test_idx]

def tabular_row_to_prompt(row):
    feature_text = ", ".join(
        f"{col}={row[col]}" for col in available_features
    )
    return (
        "You are a cybersecurity classifier. "
        "Classify this URL using only the following tabular lexical and structural features. "
        "Reply with exactly one word: phishing or legitimate.\\n"
        f"Features: {feature_text}\\n"
        "Answer:"
    )

def parse_llm_answer(text):
    text = text.lower()
    if "phishing" in text:
        return 0
    if "legitimate" in text or "benign" in text or "safe" in text:
        return 1
    return 0

llm_preds = []
llm_raw = []

for i, (_, row) in enumerate(X_llm.iterrows(), 1):
    prompt = tabular_row_to_prompt(row)
    messages = [{"role": "user", "content": prompt}]

    if tokenizer.chat_template:
        rendered = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        rendered = prompt

    inputs = tokenizer(rendered, return_tensors="pt")
    inputs = {k: v.to(llm.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    llm_raw.append(answer)
    llm_preds.append(parse_llm_answer(answer))

    if i % 20 == 0 or i == len(X_llm):
        print(f"LLM processed {i}/{len(X_llm)}")

evaluate("LLM", y_llm.to_numpy(), np.array(llm_preds), score=None)


## 14. Final six-model comparison

**Important:** the LLM is normally evaluated on a smaller subset, while the first five models use the full test set. Mention this clearly in the report.


In [ ]:
final_comparison = pd.DataFrame(results)
display(final_comparison.style.format({
    "Accuracy": "{:.4f}",
    "Precision_Phishing": "{:.4f}",
    "Recall_Phishing": "{:.4f}",
    "F1_Phishing": "{:.4f}",
    "ROC_AUC": "{:.4f}",
}))

final_comparison.to_csv("CSE445_six_model_results.csv", index=False)
files.download("CSE445_six_model_results.csv")


## 15. Save the best supervised model


In [ ]:
import joblib

supervised = {
    "Logistic Regression": lr,
    "SVM": svm,
    "Random Forest": rf,
    "XGBoost": xgb,
}

supervised_rows = pd.DataFrame(results[:4]).sort_values(
    ["Recall_Phishing", "F1_Phishing", "ROC_AUC"],
    ascending=False
)

best_name = supervised_rows.iloc[0]["Model"]
best_model = supervised[best_name]

joblib.dump(
    {"model": best_model, "features": available_features, "name": best_name},
    "best_phishing_model.joblib"
)

print("Best supervised model:", best_name)
files.download("best_phishing_model.joblib")


# Conclusion

This project compares conventional supervised tabular machine learning, an unsupervised clustering baseline, and an LLM-based approach for phishing URL detection.

For the final discussion, focus especially on **phishing recall**, because a false negative means a phishing URL was incorrectly treated as legitimate.
